In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json, math
text = open("../data/splits/train.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
print(len(chars), len(data))
val_text = open("../data/splits/val.txt", encoding="utf-8").read()
val_data = torch.tensor([stoi[c] for c in val_text if c in stoi])
print(len(val_text), len(val_data))

352 9506558
787771 787768


In [5]:
def get_batch(data, block, batch_size):
    s = torch.randint(0, len(data) - block - 1, (batch_size,))
    x = torch.stack([data[i : i + block] for i in s])
    y = torch.stack([data[i + block] for i in s])
    return x, y
torch.manual_seed(1337)
x, y = get_batch(data, 3, 32)
print(x.shape, y.shape)
print(repr("".join(itos[int(t)] for t in x[0])), "->", repr(itos[int(y[0])]))

torch.Size([32, 3]) torch.Size([32])
'ĩto' -> 'n'


In [6]:
emb = nn.Embedding(len(chars), 10)
e = emb(x)
print(e.shape)
flat = e.view(32, -1)
print(flat.shape)

torch.Size([32, 3, 10])
torch.Size([32, 30])


In [8]:
class MLP(nn.Module):
    def __init__(self, vocab_size, block, emb_dim, hidden):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.hidden = nn.Linear(block * emb_dim, hidden)
        self.out = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.shape[0], -1)
        h = torch.tanh(self.hidden(e))
        return self.out(h)
BLOCK = 3
torch.manual_seed(1337)
model = MLP(len(chars), BLOCK, 10, 200)
print(sum(p.numel() for p in model.parameters()))
logits = model(x)
print(logits.shape, F.cross_entropy(logits, y).item(), math.log(len(chars)))

80472
torch.Size([32, 352]) 5.848083972930908 5.863631175598097


In [9]:
@torch.no_grad()
def val_loss_sample(model, n=20):
    model.eval()
    losses = []
    for _ in range(n):
        xb, yb = get_batch(val_data, BLOCK, 512)
        losses.append(F.cross_entropy(model(xb), yb).item())
    model.train()
    return sum(losses) / len(losses)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
steps = 20000
for step in range(steps):
    if step == steps // 2:
        for g in opt.param_groups:
            g["lr"] = 3e-4
    xb, yb = get_batch(data, BLOCK, 64)
    loss = F.cross_entropy(model(xb), yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 2000 == 0 or step == steps - 1:
        print(step, round(loss.item(), 4), round(val_loss_sample(model), 4))


0 5.9192 5.7105
2000 2.2586 2.134
4000 1.9748 2.0971
6000 1.8727 2.0645
8000 1.6535 2.016
10000 1.971 2.081
12000 1.952 1.9446
14000 1.5519 1.9486
16000 1.9487 1.9455
18000 2.0756 1.932
19999 2.1658 1.9518
